# Legal RAG System — Interactive Demo

This notebook walks through the complete pipeline:
1. **Ingestion** — load raw legal documents
2. **Cleaning** — normalize text, fix artifacts, extract metadata
3. **Chunking** — section-aware splitting
4. **Indexing** — embed chunks into FAISS
5. **Retrieval** — hybrid dense + keyword search
6. **Generation** — grounded answer via Claude

In [ ]:
import sys, os
sys.path.insert(0, '..')  # add project root to path

# Set your API key
# os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

## Stage 1 & 2: Ingestion and Cleaning

In [ ]:
from pipeline.ingestion import load_documents
from pipeline.cleaning import clean_documents

raw_docs = load_documents('../data/raw')
print(f'Loaded {len(raw_docs)} documents:')
for d in raw_docs:
    print(f'  {d.filename} -> {d.doc_type.value}')

cleaned_docs = clean_documents(raw_docs)
print('\nCleaning stats:')
for d in cleaned_docs:
    s = d.processing_stats
    print(f'  {d.filename}: {s["original_char_count"]} -> {s["cleaned_char_count"]} chars')
    print(f'    Metadata: {d.metadata}')

## Stage 3: Chunking

In [ ]:
from pipeline.chunking import chunk_documents
import pandas as pd

chunks = chunk_documents(cleaned_docs)
print(f'Total chunks: {len(chunks)}')

df = pd.DataFrame([
    {
        'filename': c.metadata.get('filename'),
        'doc_type': c.metadata.get('doc_type'),
        'section_title': c.section_title[:50],
        'token_count': c.token_count,
        'chunk_type': c.chunk_type.value,
    }
    for c in chunks
])
print(df.groupby('filename')['token_count'].describe())
df.head(10)

## Stage 4: Vector Index

In [ ]:
from rag.vector_store import VectorStore

store = VectorStore()
store.build(chunks, verbose=True)
store.save('../data/index')

## Stage 5: Retrieval

In [ ]:
from rag.retriever import Retriever

retriever = Retriever(store, use_mmr=True)

test_queries = [
    'What are the payment terms for the license fee?',
    'Can non-compete agreements be enforced in California?',
    'What rights do consumers have under the data privacy statute?',
    'What was the court ruling on Meridian Bank?',
]

for q in test_queries:
    print(f'\nQuery: {q}')
    results = retriever.retrieve(q, k=3)
    for chunk, score in results:
        print(f'  [{score:.3f}] {chunk.metadata.get("filename")} - {chunk.section_title[:40]}')

## Stage 6: Full RAG (requires ANTHROPIC_API_KEY)

In [ ]:
from rag.rag_pipeline import LegalRAG

rag = LegalRAG()
rag.load_index('../data/index')

question = 'What are the termination rights under the software license agreement?'
response = rag.query(question, k=5)

print('Question:', response.query)
print('\nAnswer:')
print(response.answer)
print('\nSources:')
for s in response.sources:
    print(f'  - {s["filename"]} | {s["section"]} (score={s["relevance_score"]})')
print(f'\nTokens: {response.usage}')